# [Baseline] RandomForest — 풍력발전량 예측

기상 예보 데이터(LDAPS/GFS)로 KPX 발전그룹 3곳의 시간대별 **풍력발전량(kWh)** 을 예측하는 회귀 베이스라인입니다.

- **입력**: 예측 시각별 LDAPS/GFS 기상 예보 격자 평균값 + 시간·요일·계절성 캘린더 변수
- **출력**: `kpx_group_1/2/3` 세 발전그룹의 시간대별 예측 발전량(kWh)
- **모델**: 그룹별 `RandomForestRegressor` (Label 제공 기간이 그룹마다 다르므로 각각 따로 학습)

이 대회는 예측 결과 CSV(`baseline_submit.csv`)를 제출하는 방식이라, 데이터 로드부터 제출 파일 생성까지
이 노트북 하나로 처리합니다.


## 1. 라이브러리 불러오기

데이터 처리(pandas, numpy)와 회귀 모델 학습(scikit-learn)에 필요한 라이브러리를 불러옵니다.


In [49]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer


## 2. 데이터 로드

발전량 Label(`train_labels.csv`), 제출 양식(`sample_submission.csv`), LDAPS/GFS 기상 예보 데이터를 불러오고,
날짜 컬럼을 datetime으로 변환합니다. `CAPACITY_KWH` 는 그룹별 설비 용량으로, 이후 예측값을 클리핑하는 데
사용합니다.


In [50]:
DATA_DIR = Path(r"C:\Users\user\iCloudDrive\riversun\동국대학교\3학년 1학기\대외활동\DACON\데이터\open")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {
    "kpx_group_1": 21600,
    "kpx_group_2": 21600,
    "kpx_group_3": 21000,
}

train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")

ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")
ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv", encoding="utf-8-sig")
gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv", encoding="utf-8-sig")

train_labels["kst_dtm"] = pd.to_datetime(train_labels["kst_dtm"])
sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

scada_vestas = pd.read_csv(TRAIN_DIR / "scada_vestas_train.csv", encoding="utf-8-sig")
scada_unison = pd.read_csv(TRAIN_DIR / "scada_unison_train.csv", encoding="utf-8-sig")

print("train_labels:", train_labels.shape)
print("sample_submission:", sample_submission.shape)
print("ldaps_train:", ldaps_train.shape, "gfs_train:", gfs_train.shape)
print("ldaps_test:", ldaps_test.shape, "gfs_test:", gfs_test.shape)


train_labels: (26304, 4)
sample_submission: (8760, 5)
ldaps_train: (420864, 35) gfs_train: (236736, 40)
ldaps_test: (140160, 35) gfs_test: (78840, 40)


## 3. 데이터 전처리
P1. SCADA 글리치 제거

P2. Group 3 라벨 결측 처리

P3. NWP U/V → 풍속/풍향 변환

P4. Grid 선택

P5. NWP 풍속 Bias Correction

P6. Downtime 구간 플래깅

P7. Temporal Feature Engineering

P8. data_available_kst_dtm 활용 (leakage 방지)

P9. 10% 제외 규칙을 학습에 반영

P10. Train/Validation Split

In [51]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# 시간 컬럼 datetime 변환 (P10)
for df in [train_labels, scada_vestas, scada_unison, ldaps_train, gfs_train]:
    # 각 파일의 시간 컬럼명에 맞게 변환 (예: kst_dtm, forecast_kst_dtm 등)
    time_col = [col for col in df.columns if 'dtm' in col or 'time' in col][0]
    df[time_col] = pd.to_datetime(df[time_col])

# ==========================================
# P8 & 2대회 함정방지: NWP 중복 제거 및 가용성 필터링
# ==========================================
def clean_nwp(df):
    # data_available_kst_dtm이 예측대상시점(kst_dtm) 이전인 것만 필터링 (Leakage 방지)
    # ※ 컬럼명이 다를 수 있으니 확인 필요
    df = df[df['data_available_kst_dtm'] <= df['forecast_kst_dtm']]
    
    # 동일한 예측대상시점 중 '가장 최근에 발표된 예보'만 남기고 중복 제거
    df = df.sort_values(by=['forecast_kst_dtm', 'data_available_kst_dtm'])
    df = df.drop_duplicates(subset=['forecast_kst_dtm', 'grid_id'], keep='last')
    return df

print("Cleaning NWP data...")
ldaps_train = clean_nwp(ldaps_train)
gfs_train = clean_nwp(gfs_train)

# ==========================================
# P3 & P4: NWP 핵심 격자 선택 및 풍속/풍향 변환
# ==========================================
def process_nwp_features(df, prefix='ldaps'):
    # prefix에 따라 완벽하게 검증된 실제 컬럼명을 강제로 고정(하드코딩)합니다.
    if prefix == 'ldaps':
        u_col = 'heightAboveGround_10_10u'
        v_col = 'heightAboveGround_10_10v'
        t_col = 'heightAboveGround_2_t'       # 지상 2m 기온 (K)
        p_col = 'surface_0_sp'                # 지표면 기압 (Pa)
        h_col = 'heightAboveGround_2_r'       # 상대습도 (%)
    else:  # gfs인 경우
        u_col = 'heightAboveGround_10_10u'
        v_col = 'heightAboveGround_10_10v'
        t_col = 'heightAboveGround_2_2t'      # GFS 실제 지상 2m 기온 (K)
        p_col = 'surface_0_sp'                # GFS 지표면 기압 (Pa)
        h_col = 'heightAboveGround_2_2r'      # GFS 지상 2m 상대습도 (%)
    
    # ----------------------------------------------------
    # [물리 단위 영점 조절] LDAPS / GFS 공통 변환
    # ----------------------------------------------------
    # 1. 기온 보정: 켈빈(K) -> 섭씨(°C) 변환
    temp_c = df[t_col] - 273.15
    temp_k = df[t_col] # 켈빈 기온 그대로 수식 분모에 주입
    
    # 2. 기압 보정: 파스칼(Pa) -> 헥토파스칼(hPa) 변환 (두 모델 모두 10만 단위 확인 완료)
    press_hpa = df[p_col] / 100.0
    
    # ① 대기밀도 (air_density) 계산 (1위 마스터 물리 수식)
    e_s = 6.1078 * 10 ** ((7.5 * temp_c) / (temp_c + 237.3))
    e = (df[h_col] / 100.0) * e_s
    p_dry = press_hpa - e
    df['air_density'] = (p_dry / (287.058 * temp_k)) + ((e / (461.495 * temp_k)))
    
    # ② 절대습도 (absolute_humidity) 계산
    alpha = ((17.27 * temp_c) / (237.7 + temp_c)) + np.log(0.5)
    dew_point = (237.7 * alpha) / (17.27 - alpha)
    e_s_dew = 6.112 * np.exp((17.67 * dew_point) / (dew_point + 243.5))
    e_a = e_s_dew * (df[h_col] / 100.0)
    e_a_Pa = e_a * 100.0
    rho_d = (press_hpa * 100.0) / (287.058 * temp_k)
    rho_v = (e_a_Pa / (461.495 * temp_k))
    df['absolute_humidity'] = (rho_v / (rho_d + rho_v)) * 1000.0
    
    # ③ 공기분압 (air_pressure) 계산
    e_s_air = 6.112 * np.exp((17.67 * temp_c) / (temp_c + 243.5))
    e_air = e_s_air * (df[h_col] / 100.0)
    rho_d_air = (press_hpa * 100.0) / (287.058 * temp_k)
    rho_v_air = (e_air * 100.0) / (461.495 * temp_k)
    df['air_pressure'] = (rho_d_air + rho_v_air) * 287.058 * temp_k / 100.0

    # 기존 P3. 풍속/풍향 물리 변환
    df['ws'] = np.sqrt(df[u_col]**2 + df[v_col]**2)
    df['wd'] = np.arctan2(-df[u_col], -df[v_col])
    df['ws_sin'] = df['ws'] * np.sin(df['wd'])
    df['ws_cos'] = df['ws'] * np.cos(df['wd'])

    # 🎯 [신규 추가] 풍속 비선형 파생 변수 생성
    df['ws_squared'] = df['ws'] ** 2
    df['ws_cubed'] = df['ws'] ** 3
    
    # ----------------------------------------------------
    # P4. 격자 집계 (Grid-mean 전략) + 1위 신규 피처 일괄 집계
    # ----------------------------------------------------
    grouped = df.groupby('forecast_kst_dtm').agg({
        'ws': ['mean', 'max', 'std'],
        'ws_squared': ['mean', 'max'], # 🎯 제곱항 평균/최대 집계 추가
        'ws_cubed': ['mean', 'max'],   # 🎯 세제곱항 평균/최대 집계 추가
        'ws_sin': 'mean',
        'ws_cos': 'mean',
        'air_density': 'mean',       
        'absolute_humidity': 'mean', 
        'air_pressure': 'mean'       
    })
    
    # 컬럼명 정리
    grouped.columns = [f"{prefix}_{c[0]}_{c[1]}" for c in grouped.columns]
    res_df = group_data = grouped.reset_index()
    
    # 시간 컬럼 datetime 변환 보강 (타입 충돌 차단)
    res_df['forecast_kst_dtm'] = pd.to_datetime(res_df['forecast_kst_dtm'])
    return res_df

print("Extracting NWP features...")
ldaps_features = process_nwp_features(ldaps_train, prefix='ldaps')
gfs_features = process_nwp_features(gfs_train, prefix='gfs')

# ==========================================
# P1 & P6: SCADA 전처리 및 Downtime 플래깅
# ==========================================
print("Processing SCADA glitches...")
# P1. Vestas 글리치 제거 (예시: wtg01)
# 물리상한 초과만 NaN 처리 (소량 음수는 유지)
vestas_cap = 780
scada_vestas['vestas_wtg01_power_kw10m'] = np.where(
    scada_vestas['vestas_wtg01_power_kw10m'] > vestas_cap * 1.3, 
    np.nan, 
    scada_vestas['vestas_wtg01_power_kw10m']
)

# P6. Downtime 구간 플래깅 (바람은 부는데 발전량은 0 이하인 고장 구간)
scada_vestas['is_downtime'] = np.where(
    (scada_vestas['vestas_wtg01_ws'] >= 5.0) & (scada_vestas['vestas_wtg01_power_kw10m'] <= 0), 
    1, 
    0
)
# 10분 단위를 1시간 단위로 평활화할 때 downtime 비율을 피처로 활용 가능

# 🎯 [신규 추가] 10분 단위 Downtime을 1시간 단위 평활화 비율로 압축 집계
# time 컬럼을 시간 단위로 내림(floor)하여 그룹화합니다.
scada_vestas['hourly_kst'] = scada_vestas['kst_dtm'].dt.floor('h')
vestas_hourly_down = scada_vestas.groupby('hourly_kst')['is_downtime'].mean().reset_index()
vestas_hourly_down.columns = ['kst_dtm', 'vestas_downtime_ratio']

# ==========================================
# P7: Temporal Feature Engineering (시간 인코딩)
# ==========================================
def add_temporal_features(df, time_col):
    df['hour'] = df[time_col].dt.hour
    df['month'] = df[time_col].dt.month
    df['day_of_year'] = df[time_col].dt.dayofyear
    
    # Hour Cyclical Encoding
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    return df

# ==========================================
# 데이터 최종 조립 (Merge) 및 P2 보정
# ==========================================
print("Merging all features...")
# 정답 라벨 기준으로 조립 시작
final_df = train_labels.copy()

# 1. NWP 예보 피처 및 기본 시간 변수 병합
final_df = pd.merge(final_df, ldaps_features, left_on='kst_dtm', right_on='forecast_kst_dtm', how='left')
final_df = pd.merge(final_df, gfs_features, left_on='kst_dtm', right_on='forecast_kst_dtm', how='left')
final_df = add_temporal_features(final_df, 'kst_dtm')

# 2. 🎯 [3단계 추가] 1시간 단위 SCADA 다운타임 비율 병합 및 결측치 처리
final_df = pd.merge(final_df, vestas_hourly_down, on='kst_dtm', how='left')
final_df['vestas_downtime_ratio'] = final_df['vestas_downtime_ratio'].fillna(0)

# 3. 🎯 [2단계 추가] 시간 축 기준 정렬 후 시차(Lag/Lead) 및 이동 평균 피처 생성
final_df = final_df.sort_values('kst_dtm').reset_index(drop=True)

# 1) LDAPS 기반 시간 흐름 피처
final_df['ldaps_ws_mean_lag1'] = final_df['ldaps_ws_mean'].shift(1)   # 1시간 전 예보 풍속
final_df['ldaps_ws_mean_lead1'] = final_df['ldaps_ws_mean'].shift(-1) # 1시간 후 예보 풍속
final_df['ldaps_ws_rolling_3h'] = final_df['ldaps_ws_mean'].rolling(window=3, min_periods=1).mean() # 3시간 이동 평균

# 2) GFS 기반 시간 흐름 피처
final_df['gfs_ws_mean_lag1'] = final_df['gfs_ws_mean'].shift(1)
final_df['gfs_ws_mean_lead1'] = final_df['gfs_ws_mean'].shift(-1)
final_df['gfs_ws_rolling_3h'] = final_df['gfs_ws_mean'].rolling(window=3, min_periods=1).mean()

# 시프트(shift)로 발생한 끝단의 소량 결측치 처리
time_lag_cols = [
    'ldaps_ws_mean_lag1', 'ldaps_ws_mean_lead1', 'ldaps_ws_rolling_3h',
    'gfs_ws_mean_lag1', 'gfs_ws_mean_lead1', 'gfs_ws_rolling_3h'
]
final_df[time_lag_cols] = final_df[time_lag_cols].bfill().ffill()


# P2. Group 3 라벨 결측 처리 (Linear Imputation)
valid_g3 = final_df[
    final_df['kpx_group_3'].notnull() & 
    final_df['kpx_group_1'].notnull() & 
    final_df['kpx_group_2'].notnull()
]

missing_g3 = final_df[
    final_df['kpx_group_3'].isnull() & 
    final_df['kpx_group_1'].notnull() & 
    final_df['kpx_group_2'].notnull()
]

if len(missing_g3) > 0:
    print("Imputing Group 3 missing labels...")
    lr = LinearRegression()
    lr.fit(valid_g3[['kpx_group_1', 'kpx_group_2']], valid_g3['kpx_group_3'])
    pred_g3 = lr.predict(missing_g3[['kpx_group_1', 'kpx_group_2']])
    final_df.loc[missing_g3.index, 'kpx_group_3'] = pred_g3

# ==========================================
# P10: Train / Validation Split
# ==========================================
train_set = final_df[final_df['kst_dtm'].dt.year.isin([2022, 2023])]
valid_set = final_df[final_df['kst_dtm'].dt.year == 2024]

print(f"Preprocessing Done! Train shape: {train_set.shape}, Valid shape: {valid_set.shape}")

Cleaning NWP data...
Extracting NWP features...
Processing SCADA glitches...
Merging all features...
Imputing Group 3 missing labels...
Preprocessing Done! Train shape: (17519, 42), Valid shape: (8784, 42)


## 4. Feature 생성

LDAPS/GFS는 하나의 예측 시각에 여러 격자가 존재하므로, `forecast_kst_dtm`별로 수치형 기상변수의 평균값을 계산합니다.

추가로 월, 일, 시간, 요일, 주말 여부, 시간/월의 주기성을 나타내는 sin-cos feature를 생성합니다.


In [52]:
# 1. TEST 데이터도 똑같이 NWP 중복 제거 및 가용성 필터링 (P8)
print("Cleaning TEST NWP data...")
ldaps_test_clean = clean_nwp(ldaps_test)
gfs_test_clean = clean_nwp(gfs_test)

# 2. TEST 데이터도 똑같이 U/V -> 풍속/풍향 물리 변환 및 격자 집계 (P3, P4)
print("Extracting TEST NWP features...")
ldaps_test_features = process_nwp_features(ldaps_test_clean, prefix='ldaps')
gfs_test_features = process_nwp_features(gfs_test_clean, prefix='gfs')

# 3. TEST 데이터 최종 조립 (test_df 생성)
print("Merging TEST features...")
test_df = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
test_df = pd.merge(test_df, ldaps_test_features, left_on='forecast_kst_dtm', right_on='forecast_kst_dtm', how='left')
test_df = pd.merge(test_df, gfs_test_features, left_on='forecast_kst_dtm', right_on='forecast_kst_dtm', how='left')

# 4. TEST 데이터에도 동일한 시간 파생 변수 인코딩 적용 (P7)
test_df = add_temporal_features(test_df, 'forecast_kst_dtm')

# 🎯 [4번 셀 신규 추가] TEST 데이터셋에도 동일하게 시간 흐름 피처 반영
test_df = test_df.sort_values('forecast_kst_dtm').reset_index(drop=True)
test_df['ldaps_ws_mean_lag1'] = test_df['ldaps_ws_mean'].shift(1)
test_df['ldaps_ws_mean_lead1'] = test_df['ldaps_ws_mean'].shift(-1)
test_df['ldaps_ws_rolling_3h'] = test_df['ldaps_ws_mean'].rolling(window=3, min_periods=1).mean()

test_df['gfs_ws_mean_lag1'] = test_df['gfs_ws_mean'].shift(1)
test_df['gfs_ws_mean_lead1'] = test_df['gfs_ws_mean'].shift(-1)
test_df['gfs_ws_rolling_3h'] = test_df['gfs_ws_mean'].rolling(window=3, min_periods=1).mean()

test_df[time_lag_cols] = test_df[time_lag_cols].bfill().ffill()

# 🎯 [4번 셀 신규 추가] TEST 세트에는 다운타임 피처를 0으로 채워 구조적 결합 유지
test_df['vestas_downtime_ratio'] = 0.0

# 5. 모델 학습에 사용할 최종 피처(X)와 라벨(y) 확정 분리
drop_features = ["kst_dtm", "forecast_kst_dtm", "forecast_id", *TARGET_COLS]

# [TRAIN / VALID] - 3번 셀의 시계열 스플릿 결과물 활용 (P10)
X_train = train_set.drop(columns=drop_features, errors='ignore')
y_train_total = train_set[TARGET_COLS]

X_val = valid_set.drop(columns=drop_features, errors='ignore')
y_val_total = valid_set[TARGET_COLS]

# [TEST] - 7번 제출 파일 생성을 위해 모델에 들어갈 최종 테스트 피처
X_test = test_df.drop(columns=drop_features, errors='ignore')

print("\n--- 파이프라인 조립 완료 ---")
print("X_train (학습 피처):", X_train.shape)
print("X_val   (검증 피처):", X_val.shape)
print("X_test  (평가 피처):", X_test.shape)

Cleaning TEST NWP data...
Extracting TEST NWP features...
Merging TEST features...

--- 파이프라인 조립 완료 ---
X_train (학습 피처): (17519, 38)
X_val   (검증 피처): (8784, 38)
X_test  (평가 피처): (8760, 36)


In [53]:
# # === 4.5번 셀: 변수명 완벽 격리 + 시드 고정 버전 ===
# import optuna
# from optuna.samplers import TPESampler
# import lightgbm as lgb
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.impute import SimpleImputer

# print("🔥 [ALL TARGETS] 대회 산식(FICR) 최대화 변수 격리 튜닝 시작...")

# imputer = SimpleImputer(strategy="median")
# X_train_numeric = X_train.select_dtypes(include=[np.number])
# X_train_imp_local = pd.DataFrame(imputer.fit_transform(X_train_numeric), columns=X_train_numeric.columns)

# best_params_dict = {}

# def calculate_custom_ficr(y_true, y_pred, capacity):
#     eval_mask = y_true >= (capacity * 0.1)
#     if eval_mask.sum() == 0:
#         return 0
#     y_true_eval = y_true[eval_mask]
#     y_pred_eval = y_pred[eval_mask]
    
#     absolute_errors = np.abs(y_pred_eval - y_true_eval)
#     hourly_nmae = absolute_errors / capacity
    
#     revenue = np.where(hourly_nmae <= 0.06, 4.0, 
#                        np.where(hourly_nmae <= 0.08, 3.0, 0.0))
#     max_revenue = len(y_true_eval) * 4.0
#     return revenue.sum() / max_revenue

# for target_col in TARGET_COLS:
#     print(f"\n🎯 [{target_col}] 파라미터 사냥 중...")
    
#     train_mask = train_set[target_col].notna()
#     X_opt = X_train_imp_local.loc[train_mask]
#     y_opt = train_set.loc[train_mask, target_col].values
    
#     # 🎯 [핵심] X_val 변수명이 오염되지 않도록 뒤에 _opt를 붙여 철저히 분리합니다.
#     X_tr_opt, X_val_opt, y_tr_opt, y_val_opt = train_test_split(X_opt, y_opt, test_size=0.2, random_state=42)
#     target_capacity = CAPACITY_KWH[target_col]
    
#     def objective(trial):
#         params = {
#             'n_estimators': trial.suggest_int('n_estimators', 400, 1200, step=50),
#             'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.06, log=True),
#             'max_depth': trial.suggest_int('max_depth', 6, 12),
#             'num_leaves': trial.suggest_int('num_leaves', 31, 127),
#             'min_child_samples': trial.suggest_int('min_child_samples', 15, 35),
#             'subsample': trial.suggest_float('subsample', 0.65, 0.95),
#             'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 0.95),
#             'random_state': 42,
#             'n_jobs': -1,
#             'verbose': -1
#         }
        
#         model = lgb.LGBMRegressor(**params)
#         model.fit(X_tr_opt, y_tr_opt)
        
#         preds = model.predict(X_val_opt)
#         preds = np.clip(preds, 0, target_capacity)
        
#         ficr_score = calculate_custom_ficr(y_val_opt, preds, target_capacity)
#         return 1.0 - ficr_score

#     fixed_sampler = TPESampler(seed=42)
#     study = optuna.create_study(direction='minimize', sampler=fixed_sampler)
#     study.optimize(objective, n_trials=20)
    
#     best_params_dict[target_col] = study.best_trial.params
#     print(f"▶ [{target_col}] 최고 가상 FICR: {1.0 - study.best_value:.5f}")

# print("\n🏆 모든 타겟 격리 사냥 완료!")

## 5. 모델 학습

KPX 그룹별로 Label 제공 기간이 다르므로, 각 그룹의 Label이 존재하는 행만 사용해 RandomForest 모델을 따로 학습합니다.

RandomForest는 여러 결정트리를 앙상블하는 배깅(Bagging) 모델로, 각 트리를 부트스트랩 샘플과 랜덤 피처
subset으로 학습한 뒤 예측을 평균 냅니다.


In [54]:
# === 5번 셀(cell09) 실시간 Fold별 스코어 출력 + 모델 저장 버전 ===
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

# Fold별 로컬 점수 계산용 내부 함수
def calculate_fold_ficr(y_true, y_pred, capacity):
    valid = y_true >= (capacity * 0.10)
    if np.sum(valid) == 0: return 0.0, 0.0
    actual = y_true[valid]
    forecast = y_pred[valid]
    
    error_rate = np.abs(forecast - actual) / capacity
    nmae_score = 1.0 - np.mean(error_rate)
    
    unit_price = np.select([error_rate <= 0.06, error_rate <= 0.08], [4.0, 3.0], default=0.0)
    ficr_score = np.sum(actual * unit_price) / np.sum(actual * 4.0)
    return nmae_score, ficr_score

# 1. 수치형 데이터 임퓨팅 (무결점 데이터 베이스)

X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])
# 🎯 [핵심 보정] X_train의 수치형 컬럼 순서와 일치하도록 X_test 컬럼 순서 강제 동기화
# 혹시 test에 없는 컬럼이 있다면 NaN으로 채우고, 순서를 정확히 맞춥니다.
X_test_numeric = X_test_numeric.reindex(columns=X_train_numeric.columns)
imputer = SimpleImputer(strategy="median")

X_train_imp = pd.DataFrame(imputer.fit_transform(X_train_numeric), columns=X_train_numeric.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test_numeric), columns=X_test_numeric.columns)

final_test_predictions = pd.DataFrame(index=sample_submission.index)
oof_predictions = pd.DataFrame(0.0, index=train_set.index, columns=TARGET_COLS)
models_dict = {target: [] for target in TARGET_COLS}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

print("🚀 5-Fold Cross Validation Training Started...")

for target in TARGET_COLS:
    print(f"\n==================== 🎯 TARGET: {target} ====================")
    
    train_mask = train_set[target].notna()
    X_target = X_train_imp.loc[train_mask].reset_index(drop=True)
    y_target = train_set.loc[train_mask, target].reset_index(drop=True)
    target_indices = train_set.loc[train_mask].index
    
    target_capacity = CAPACITY_KWH[target]
    test_pred_folds = np.zeros(len(X_test_imp))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_target, y_target)):
        X_tr, y_tr = X_target.iloc[train_idx], y_target.iloc[train_idx]
        X_va, y_va = X_target.iloc[val_idx], y_target.iloc[val_idx]
        
        threshold = target_capacity * 0.1
        sample_weights = np.where(y_tr < threshold, 0.1, 1.0)
        
        model = lgb.LGBMRegressor(
            n_estimators=1050, learning_rate=0.04571276682924094, max_depth=12,
            num_leaves=108, min_child_samples=15, subsample=0.8614710907817068,
            colsample_bytree=0.8210216619560089, random_state=42, n_jobs=-1, verbose=-1
        )
        
        model.fit(X_tr, y_tr, sample_weight=sample_weights)
        models_dict[target].append(model)
        
        val_preds = model.predict(X_va)
        val_preds = np.clip(val_preds, 0, target_capacity)
        
        actual_val_indices = target_indices[val_idx]
        oof_predictions.loc[actual_val_indices, target] = val_preds
        
        fold_test_pred = model.predict(X_test_imp)
        test_pred_folds += np.clip(fold_test_pred, 0, target_capacity) / kf.n_splits
        
        # 🎯 [실시간 출력] 현재 Fold의 검증 점수 산출 및 출력
        f_nmae, f_ficr = calculate_fold_ficr(y_va.to_numpy(), val_preds, target_capacity)
        print(f"▶ Fold {fold+1} Done! | [OOF] 1-NMAE: {f_nmae:.4f}, FICR: {f_ficr:.4f} (Total: {0.5*f_nmae + 0.5*f_ficr:.4f})")
        
    final_test_predictions[target] = test_pred_folds

print("\n==================================================")
print("🏆 5-Fold 교차 검증 최종 OOF 스코어 리포트 산출 완료!")

🚀 5-Fold Cross Validation Training Started...

==================== 🎯 TARGET: kpx_group_1 ====================
▶ Fold 1 Done! | [OOF] 1-NMAE: 0.9082, FICR: 0.5103 (Total: 0.7092)
▶ Fold 2 Done! | [OOF] 1-NMAE: 0.9103, FICR: 0.5279 (Total: 0.7191)
▶ Fold 3 Done! | [OOF] 1-NMAE: 0.9066, FICR: 0.5171 (Total: 0.7119)
▶ Fold 4 Done! | [OOF] 1-NMAE: 0.9106, FICR: 0.5157 (Total: 0.7132)
▶ Fold 5 Done! | [OOF] 1-NMAE: 0.9115, FICR: 0.5343 (Total: 0.7229)

==================== 🎯 TARGET: kpx_group_2 ====================
▶ Fold 1 Done! | [OOF] 1-NMAE: 0.9064, FICR: 0.5368 (Total: 0.7216)
▶ Fold 2 Done! | [OOF] 1-NMAE: 0.9043, FICR: 0.5096 (Total: 0.7070)
▶ Fold 3 Done! | [OOF] 1-NMAE: 0.9057, FICR: 0.5245 (Total: 0.7151)
▶ Fold 4 Done! | [OOF] 1-NMAE: 0.9088, FICR: 0.5327 (Total: 0.7208)
▶ Fold 5 Done! | [OOF] 1-NMAE: 0.9095, FICR: 0.5468 (Total: 0.7281)

==================== 🎯 TARGET: kpx_group_3 ====================
▶ Fold 1 Done! | [OOF] 1-NMAE: 0.9073, FICR: 0.5146 (Total: 0.7110)
▶ Fold 2 Do

## 6. 테스트 데이터 예측 생성

학습한 그룹별 모델로 평가 기간 전체의 발전량을 예측합니다. 예측값은 0 이상, 설비 용량 이하로 클리핑하여
물리적으로 불가능한 값을 방지합니다.


In [55]:
# === 6번 셀: 테스트 데이터 예측 생성 (변수명 보정 완료) ===
submission = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()

for target in TARGET_COLS:
    # 🎯 [핵심 보정] 5-Fold 교차 검증의 평균값인 final_test_predictions를 매칭합니다.
    submission[target] = final_test_predictions[target]

submission["forecast_kst_dtm"] = pd.to_datetime(submission["forecast_kst_dtm"]).dt.strftime("%Y-%m-%d %H:%M:%S")

print(submission.head())
print(submission.shape)

     forecast_id     forecast_kst_dtm   kpx_group_1   kpx_group_2  \
0  forecast_0001  2025-01-01 01:00:00  16133.649679  19402.944485   
1  forecast_0002  2025-01-01 02:00:00  16442.711006  18946.931259   
2  forecast_0003  2025-01-01 03:00:00  16982.676267  19209.809305   
3  forecast_0004  2025-01-01 04:00:00  17314.375154  19432.103779   
4  forecast_0005  2025-01-01 05:00:00  17078.374532  18838.181005   

    kpx_group_3  
0  16165.024917  
1  15890.729220  
2  16712.607546  
3  16407.279295  
4  16961.178603  
(8760, 5)


## 7. 평가 산식

In [56]:
# === 7번 셀: 평가 산식 (5-Fold 앙상블 예측 보정 완료) ===

# 데이콘 공식 메트릭 함수 정의
def metric(answer_df, pred_df):
    group_nmae = []
    group_ficr = []

    for col in TARGET_COLS:
        actual = answer_df[col].to_numpy(dtype=float)
        forecast = pred_df[col].to_numpy(dtype=float)
        capacity = CAPACITY_KWH[col]

        valid = actual >= capacity * 0.10

        actual = actual[valid]
        forecast = forecast[valid]

        error_rate = np.abs(forecast - actual) / capacity
        group_nmae.append(np.mean(error_rate))

        unit_price = np.select(
            [error_rate <= 0.06, error_rate <= 0.08],
            [4.0, 3.0],
            default=0.0,
        )

        earned_settlement = np.sum(actual * unit_price)
        max_settlement = np.sum(actual * 4.0)

        group_ficr.append(earned_settlement / max_settlement)

    one_minus_nmae = 1 - np.mean(group_nmae)
    ficr = np.mean(group_ficr)

    total_score = 0.5 * one_minus_nmae + 0.5 * ficr

    return total_score, one_minus_nmae, ficr


# ----------------------------------------------------
# 2024년 검증 데이터(X_val)에 대한 로컬 점수 계산 시작
# ----------------------------------------------------
print("Calculating Local Validation Score with 5-Fold Ensemble...")

# 1. 실제 2024년의 정답 라벨 데이터 준비 (Y)
answer_df = valid_set[TARGET_COLS].copy()

# 2. 5번 셀에서 학습이 완료된 5개 Fold 모델들의 평균으로 2024년 검증 피처 예측값 생성 (Y_pred)
X_val_numeric = X_val.select_dtypes(include=[np.number])
X_val_imp = pd.DataFrame(imputer.transform(X_val_numeric), columns=X_val_numeric.columns)

pred_df = pd.DataFrame(index=valid_set.index)

for target in TARGET_COLS:
    target_capacity = CAPACITY_KWH[target]
    
    # 🎯 5개 Fold 모델의 예측값을 누적할 배열 초기화
    val_ensemble_preds = np.zeros(len(X_val_imp))
    
    # 저장해둔 5개의 모델을 꺼내와서 소프트 보팅(평균)
    for model in models_dict[target]:
        val_ensemble_preds += model.predict(X_val_imp) / len(models_dict[target])
        
    # 물리적 상하한 클리핑 적용 후 저장
    pred_df[target] = np.clip(val_ensemble_preds, 0, target_capacity)

# 3. 점수 산출
total_score, one_minus_nmae, ficr = metric(answer_df, pred_df)

print("\n==========================================")
print("🎯 LOCAL VALIDATION SCORE REPORT (2024)")
print("==========================================")
print(f"▶ 최종 점수 (Total Score) : {total_score:.5f}")
print(f"▶ 1-NMAE                 : {one_minus_nmae:.5f}")
print(f"▶ 정산금획득률 (FICR)     : {ficr:.5f}")
print("==========================================")

Calculating Local Validation Score with 5-Fold Ensemble...

🎯 LOCAL VALIDATION SCORE REPORT (2024)
▶ 최종 점수 (Total Score) : 0.58982
▶ 1-NMAE                 : 0.86295
▶ 정산금획득률 (FICR)     : 0.31669


## 8. baseline_submit.csv 생성

`sample_submission.csv` 형식에 맞춰 최종 제출 파일을 생성합니다. 이 대회는 결과 CSV를 직접 제출하는
방식이므로, 이 파일을 그대로 제출하면 됩니다.


In [57]:
output_path = DATA_DIR / "baseline_submit.csv"
submission.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Saved: {output_path}")


Saved: C:\Users\user\iCloudDrive\riversun\동국대학교\3학년 1학기\대외활동\DACON\데이터\open\baseline_submit.csv
